# Build `Evaluation Results Database`

### `geometric` & `radiometric`

Paul Montesano, PhD  
June-Sept 2026

In [80]:
options(warn=-1)
library(tidyverse)
library(patchwork)
library(readr)

```
CSDA_eval/results/evaluation_results_db
 └─ acquisitions.csv             # All acquisitions use across all CSDA optical evaluations
 └─ radiometric_db
   ├── results_snr.csv           # SNR metrics
   ├── results_abscal.csv        # absolute calibration accuracy metrics  
   └── results_tempstab.csv      # temporal stability metrics
 └─ geometric_db             
   ├── results_apa.csv           # abs position accuracy metrics
   ├── results_ssr.csv           # sensor spatial response metrics 
   ├── results_bbr.csv           # band-to-band ratio metrics  
   └── results_tempstab.csv      # temporal stability metrics

In [28]:
DIR_EVAL_RESULTS_DB = "/explore/nobackup/projects/CSDA_eval/results/evaluation_results_db"

### Find all csvs

In [29]:
# Define the root search path
search_path <- "/explore/nobackup/projects/CSDA_eval/results"

# 1. Grab ALL CSV files recursively down the tree
exclude_strings <- c('overview', 'smry', 'RSR','test','evaluation_results_db')  # Extend as needed
exclude_pattern <- paste(exclude_strings, collapse = "|")

all_csvs <- list.files(
  path = search_path,
  pattern = "\\.csv$",    # Regular expression matching strings that end with .csv
  recursive = TRUE,
  full.names = TRUE
) %>%
  .[!grepl(exclude_pattern, basename(.), ignore.case = TRUE)]  # ignore.case for robustness

print(glue::glue("Found {length(all_csvs)} CSV files"))
print(glue::glue("Excluded pattern: '{exclude_pattern}'"))

Found 61 CSV files
Excluded pattern: 'overview|smry|RSR|test|evaluation_results_db'


In [30]:
read_evaluation_files <- function(files_list){

    ACQ_COLS <- c(
      'Evaluation_type', 'Vendor', 'Constellation', 'Satellite',
      'Site_name', 'Product_level',
      'Acq_Datetime', 
        #'date', 
        'source_file'
        )
    
    # Band order defined inside function so it's always available
    band_order <- c(
        'Pan', 'Coastal', 'CB', 'Blue', 'B', 'Green', 'G', 'Yellow', 'Y', 
        'Red', 'R', 'RedEdge', 'RE', 'RedEdge1', 'RE1', 'RedEdge2', 'RE2', 
        'NIR', 'NIR1', 'NIR2', 'SWIR1', 'SWIR2'
    )

    df = files_list %>%
            set_names() %>%
            map_df(read_csv, .id = "source_file", show_col_types=FALSE) %>%
            
            # Drop empty unnamed trailing columns
            select(-matches("^\\.\\.\\.\\d+$")) %>%
            select(where(~ !all(is.na(.x)))) %>%
            
            # Standardize column names
            rename_with(~ case_when(
                .x == "Evaluation Type" ~ "Evaluation_type",
                TRUE                    ~ .x
            )) %>%
            
            # # Add missing common acquisition cols as NA
            # { if (!"Date_time_in_UTC(YYYYMMDD_HHMMSS)" %in% names(.))
            #     mutate(., `Date_time_in_UTC(YYYYMMDD_HHMMSS)` = NA_character_) else . } %>%
            # { if (!"Time_series_date_min(YYYYMMDD)" %in% names(.))
            #     mutate(., `Time_series_date_min(YYYYMMDD)` = NA_character_) else . } %>%
            # { if (!"Time_series_date_max(YYYYMMDD)" %in% names(.))
            #     mutate(., `Time_series_date_max(YYYYMMDD)` = NA_character_) else . } %>%
            { if (!"Acq_Datetime" %in% names(.))
                mutate(., Acq_Datetime = NA_character_) else . } %>%
            { if (!"Acq_ID" %in% names(.))
                mutate(., Acq_ID = NA_character_) else . } %>%
            { if (!"Satellite" %in% names(.))
                mutate(., Satellite = NA_character_) else . } %>%
            { if (!"Band_name" %in% names(.))
                mutate(., Band_name = NA_character_) else . } %>%
            
            mutate(
                # KEY FIX: convert Acq_Datetime to character first
                # so it's always a string going into parse_date_time
                # regardless of how read_csv interpreted it
                Acq_Datetime = case_when(
                    !is.na(Acq_Datetime) ~
                        parse_date_time(as.character(Acq_Datetime),
                                       orders = c('YmdHMS', 'Ymd HMS',
                                                  'Y-m-d H:M:S', 'Y-m-d',
                                                  'Ymd',
                                                  'mdy',        # ADD: handles 11/3/2024
                                                  'mdY',        # ADD: handles 11/03/2024
                                                  'dmy'),       # ADD: handles European format
                                       tz = 'UTC'),
                    !is.na(Acq_Datetime) ~
                        parse_date_time(Acq_Datetime,
                                       orders = 'YmdHMS',
                                       tz = 'UTC'),
                    TRUE ~ NA_POSIXct_
                ),
                #date = as.Date(Acq_Datetime),
                # Time_series_date_min = as.Date(
                #     as.character(`Time_series_date_min(YYYYMMDD)`),
                #     format = '%Y%m%d'
                # ),
                # Time_series_date_max = as.Date(
                #     as.character(`Time_series_date_max(YYYYMMDD)`),
                #     format = '%Y%m%d'
                # ),
                Satellite = as.character(Satellite),
                .keep = "unused"
            ) %>%
            mutate(
                Band_name = factor(Band_name, levels = band_order),
                Satellite = factor(Satellite)
            ) %>%
            relocate(any_of(ACQ_COLS), .before = everything()) %>%
            relocate(source_file, .after = last_col()) %>%
            filter(if_any(-source_file, ~ !is.na(.))) # Removes extra rows that are just NA

    return(df)
}

### Radiometric results csvs

In [31]:
snr_files <-         grep("/radiometric/snr/.*_snr_eachcase\\.csv$", all_csvs, value = TRUE)
abscal_files <-      grep("/radiometric/abscal/.*_abscal_eachcase\\.csv$", all_csvs, value = TRUE)
tempstabrad_files <- grep("/radiometric/tempstab/.*_tempstab_summary.csv$", all_csvs, value = TRUE)

In [38]:
# Usage - each file type read separately
df_snr      <- read_evaluation_files(snr_files)
df_abscal      <- read_evaluation_files(abscal_files)
df_tempstabrad      <- read_evaluation_files(tempstabrad_files)

New names:
• `Radiometric_Temporal_Stability_Gain_Surface_reflectance_against_MODISBRF(%/year)`
  ->
  `Radiometric_Temporal_Stability_Gain_Surface_reflectance_against_MODISBRF(%/year)...24`
• `Radiometric_Temporal_Stability_Gain_Surface_reflectance_against_MAIACAC(coeff_of_variation%)`
  ->
  `Radiometric_Temporal_Stability_Gain_Surface_reflectance_against_MAIACAC(coeff_of_variation%)...25`
• `Radiometric_Temporal_Stability_Gain_Surface_reflectance_against_MODISBRF(%/year)`
  ->
  `Radiometric_Temporal_Stability_Gain_Surface_reflectance_against_MODISBRF(%/year)...26`
• `Radiometric_Temporal_Stability_Gain_Surface_reflectance_against_MAIACAC(coeff_of_variation%)`
  ->
  `Radiometric_Temporal_Stability_Gain_Surface_reflectance_against_MAIACAC(coeff_of_variation%)...27`
New names:
• `Radiometric_Temporal_Stability_Gain_Surface_reflectance_against_MODISBRF(%/year)`
  ->
  `Radiometric_Temporal_Stability_Gain_Surface_reflectance_against_MODISBRF(%/year)...24`
• `Radiometric_Temporal_Stabil

In [57]:
DIR_RADIOMETRIC_DB = file.path(DIR_EVAL_RESULTS_DB, "radiometric_db")
dir.create(DIR_RADIOMETRIC_DB, showWarnings = FALSE, recursive = TRUE)
setwd(DIR_RADIOMETRIC_DB)

In [58]:
write_csv(df_snr, "results_snr.csv")
write_csv(df_abscal, "results_abscal.csv")
write_csv(df_tempstabrad, "results_tempstab.csv")

### Geometric results csvs

In [59]:
# 2. Filter down to only files that live inside an 'apa' subdirectory
apa_files <-         grep("/geometric/apa/.*09012026\\.csv$", all_csvs, value = TRUE)
bbr_files <-         grep("/geometric/bbr/.*09012026\\.csv$", all_csvs, value = TRUE)
ssr_files <-         grep("/geometric/ssr/.*09012026\\.csv$", all_csvs, value = TRUE)
tempstabgeo_files <- grep("/geometric/tempstab/.*09012026\\.csv$", all_csvs, value = TRUE)

In [60]:
# Usage - each file type read separately
df_apa      <- read_evaluation_files(apa_files)
df_ssr      <- read_evaluation_files(ssr_files)
df_bbr      <- read_evaluation_files(bbr_files)
df_tempstabgeo      <- read_evaluation_files(tempstabgeo_files)

Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `Acq_Datetime = case_when(...)`.
Caused by warning:
! All formats failed to parse. No formats found.”
New names:
• `` -> `...17`
• `` -> `...18`
• `` -> `...19`
• `` -> `...20`
• `` -> `...21`
• `` -> `...22`
Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `Acq_Datetime = case_when(...)`.
Caused by warning:
! All formats failed to parse. No formats found.”
Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `Acq_Datetime = case_when(...)`.
Caused by warning:
! All formats failed to parse. No formats found.”


In [63]:
DIR_GEOMETRIC_DB = file.path(DIR_EVAL_RESULTS_DB, "geometric_db")
dir.create(DIR_GEOMETRIC_DB, showWarnings = FALSE, recursive = TRUE)
setwd(DIR_GEOMETRIC_DB)

In [64]:
write_csv(df_apa, "results_apa.csv")
write_csv(df_ssr, "results_ssr.csv")
write_csv(df_bbr, "results_bbr.csv")
write_csv(df_tempstabgeo, "results_tempstab.csv")

In [65]:
head(df_apa)

Evaluation_type,Vendor,Constellation,Satellite,Site_name,Product_level,Acq_Datetime,Acq_ID,ref_band,n_chips,xoffset_m,yoffset_m,xstd_m,ystd_m,xrmse_m,yrmse_m,ce90_m,ce90-demean_m,Band_name,source_file
<chr>,<chr>,<chr>,<fct>,<chr>,<chr>,<dttm>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<chr>
apa,vantor,legion,1,belo_horizonte,S3DS,2024-10-06,24OCT06130617-S3DS_R5C1-200010869547_01_P001,1,80,0.754103,-0.108310,0.570141,1.899454,0.943223,1.890650,1.110162,0.808986,NA,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/apa/LEG-belo_horizonte-apa-09012026.csv
apa,vantor,legion,1,belo_horizonte,S3DS,2024-10-06,24OCT06130617-S3DS_R5C2-200010869547_01_P001,1,60,0.801057,0.304296,0.349616,0.183214,0.872861,0.354406,1.150507,0.409957,NA,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/apa/LEG-belo_horizonte-apa-09012026.csv
apa,vantor,legion,1,belo_horizonte,S3DS,2024-10-06,24OCT06130617-S3DS_R6C1-200010869547_01_P001,1,213,0.844994,-0.062115,0.307379,0.217993,0.898918,0.226177,1.230074,0.598724,NA,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/apa/LEG-belo_horizonte-apa-09012026.csv
apa,vantor,legion,1,belo_horizonte,S3DS,2024-10-06,24OCT06130617-S3DS_R6C2-200010869547_01_P001,1,172,0.831803,0.377818,0.220860,0.182972,0.860461,0.419560,1.245892,0.444095,NA,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/apa/LEG-belo_horizonte-apa-09012026.csv
apa,vantor,legion,1,belo_horizonte,S3DS,2024-10-06,24OCT06130617-S3DS_R7C1-200010869547_01_P001,1,185,0.945622,0.085211,0.228758,0.307854,0.972753,0.318626,1.307971,0.576325,NA,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/apa/LEG-belo_horizonte-apa-09012026.csv
apa,vantor,legion,1,belo_horizonte,S3DS,2024-10-06,24OCT06130617-S3DS_R7C2-200010869547_01_P001,1,148,1.087873,0.530414,0.728809,0.395686,1.308068,0.660945,1.993111,1.081225,NA,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/apa/LEG-belo_horizonte-apa-09012026.csv


In [66]:
head(df_bbr)

Evaluation_type,Vendor,Constellation,Satellite,Site_name,Product_level,Acq_Datetime,ref_band,test_band,n_chips,⋯,yoffset_m,xstd_m,ystd_m,xrmse_m,yrmse_m,ce90_m,ce90-demean_m,Acq_ID,Band_name,source_file
<chr>,<chr>,<chr>,<fct>,<chr>,<chr>,<dttm>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<fct>,<chr>
bbr,vantor,legion,NA,all_geometric,S3DS,NA,1,2,596205,⋯,0.000402,0.010446,0.010437,0.010446,0.010445,0.014018,0.013989,NA,NA,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/bbr/bbr_3band_LEG-09012026.csv
bbr,vantor,legion,NA,all_geometric,S3DS,NA,1,3,745713,⋯,0.000026,0.010357,0.016714,0.010400,0.016714,0.021762,0.021700,NA,NA,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/bbr/bbr_3band_LEG-09012026.csv
bbr,vantor,legion,NA,all_geometric,S3DS,NA,2,3,648023,⋯,0.000431,0.014906,0.020117,0.014941,0.020122,0.022085,0.022001,NA,NA,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/bbr/bbr_3band_LEG-09012026.csv
bbr,vantor,legion,NA,all_geometric,M3DS,NA,1,2,18158,⋯,0.014602,0.062253,0.055806,0.062592,0.057683,0.070586,0.066026,NA,NA,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/bbr/bbr_8band_LEG-09012026.csv
bbr,vantor,legion,NA,all_geometric,M3DS,NA,1,3,30393,⋯,0.112285,0.219338,0.221424,0.244036,0.248264,0.670581,0.515722,NA,NA,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/bbr/bbr_8band_LEG-09012026.csv
bbr,vantor,legion,NA,all_geometric,M3DS,NA,1,4,21268,⋯,0.136933,0.218187,0.220715,0.257478,0.259737,0.687224,0.494374,NA,NA,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/bbr/bbr_8band_LEG-09012026.csv


### Create acquisitions table

In [70]:
# ============================================================
# COMMON COLS FOR ACQUISITIONS TABLE
# ============================================================
common_cols <- c(
  'Evaluation_type', 'Vendor', 'Constellation', 'Satellite',
  'Site_name', 'Product_level',
  'Acq_Datetime', 
    #'date', 
    'source_file'
)

# ============================================================
# BUILD ACQUISITIONS TABLE
# Collapse band rows -> one row per unique Acq_Datetime
# (not uding TempStab)
# ============================================================
acquisitions <- bind_rows(
    # Radiometric dB
    df_snr %>% select(any_of(common_cols), Band_name),
    df_abs %>% select(any_of(common_cols), Band_name),
    # Geometric dB
    df_apa %>% select(any_of(common_cols), Band_name),
    df_ssr %>% select(any_of(common_cols), Band_name),
    df_bbr %>% select(any_of(common_cols), Band_name),
  ) %>%
  group_by(across(all_of(common_cols))) %>%
  summarise(
    bands      = list(sort(unique(as.character(Band_name)))),  # Collapsed band list
    n_bands    = n_distinct(Band_name),
    .groups    = 'drop'
  ) %>%
  # Unique acquisitions by Acq_Datetime
  distinct(Acq_Datetime, .keep_all = TRUE) %>%
  arrange(Acq_Datetime) %>%
  relocate(source_file, .after = everything()) %>%
  #mutate(acq_id = row_number()) %>%
  #relocate(acq_id, .before = everything()) %>%
  #mutate(Evaluation_category = 'Radiometric') %>%
  #relocate(Evaluation_category, .before = everything()) %>%
  as.data.frame()

setwd(DIR_EVAL_RESULTS_DB)
write_csv(acquisitions, 'acquisitions.csv')

### Make readable

In [79]:
system(paste("chmod -R 755", shQuote(DIR_EVAL_RESULTS_DB)))

In [72]:
# Preview
acquisitions %>%
  select(
      #acq_id, 
      'Vendor', 'Constellation', 'Satellite', Acq_Datetime, n_bands, bands) %>%
  head(3)

print(glue::glue("CSDA Evaluation Acquisitions: {nrow(acquisitions)} unique acquisitions"))

,Vendor,Constellation,Satellite,Acq_Datetime,n_bands,bands
,<chr>,<chr>,<fct>,<dttm>,<int>,<list>
1,Satellogic,NewSat,SN10,2021-02-02 09:01:41,4,"Blue , Green, NIR , Red"
2,Satellogic,NewSat,SN10,2021-02-05 09:10:47,4,"Blue , Green, NIR , Red"
3,Satellogic,NewSat,SN10,2021-02-12 09:00:48,4,"Blue , Green, NIR , Red"


CSDA Evaluation Acquisitions: 104 unique acquisitions


In [23]:
# TODO: need to change source bbr tables to be site specific 
tail(acquisitions %>% filter(Site_name == 'all_geometric'))

,acq_id,Evaluation_type,Vendor,Constellation,Satellite,Site_name,Product_level,Acq_Datetime,date,bands,n_bands,source_file
,<int>,<chr>,<chr>,<chr>,<fct>,<chr>,<chr>,<dttm>,<date>,<list>,<int>,<chr>
1,104,bbr,vantor,legion,NA,all_geometric,M3DS,NA,NA,,1,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/bbr/bbr_8band_LEG-09012026.csv


In [74]:
tail(acquisitions)

,Evaluation_type,Vendor,Constellation,Satellite,Site_name,Product_level,Acq_Datetime,bands,n_bands,source_file
,<chr>,<chr>,<chr>,<fct>,<chr>,<chr>,<dttm>,<list>,<int>,<chr>
99,apa,vantor,legion,5,melbourne,S3DS,2025-11-25 00:00:00,,1,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/apa/LEG-melbourne-apa-09012026.csv
100,Radiometric,Airbus,PleiadesNeo,PNEO3,PICS Libya-4,Basic,2026-02-25 09:07:49,"Blue , Coastal, Green , NIR , Red , RedEdge",6,/explore/nobackup/projects/CSDA_eval/results/airbus/pleiadesneo/radiometric/snr/Radiometric_evaluation_Airbus_PleiadesNeo_snr_eachcase.csv
101,Radiometric,Airbus,Pleiades,PHR1A,PICS Libya-4,Basic,2026-02-25 09:21:42,"Blue , Green, NIR , Red",4,/explore/nobackup/projects/CSDA_eval/results/airbus/pleiades/radiometric/abscal/Radiometric_evaluation_Airbus_Pleiades_abscal_eachcase.csv
102,Radiometric,Airbus,SPOT,SPOT6,PICS Libya-4,Basic,2026-02-26 08:46:49,"Blue , Green, NIR , Red",4,/explore/nobackup/projects/CSDA_eval/results/airbus/spot6/radiometric/abscal/Radiometric_evaluation_Airbus_SPOT_abscal_eachcase.csv
103,Radiometric,Airbus,PleiadesNeo,PNEO4,PICS Libya-4,Basic,2026-03-05 09:11:35,"Blue , Coastal, Green , NIR , Red , RedEdge",6,/explore/nobackup/projects/CSDA_eval/results/airbus/pleiadesneo/radiometric/snr/Radiometric_evaluation_Airbus_PleiadesNeo_snr_eachcase.csv
104,bbr,vantor,legion,NA,all_geometric,M3DS,NA,,1,/explore/nobackup/projects/CSDA_eval/results/maxar/legion/geometric/bbr/bbr_8band_LEG-09012026.csv


In [75]:
# # ============================================================
# # BUILD RESULTS TABLES (evaluation-specific metrics)
# # ============================================================

# # Helper to add acq_id by joining on common cols
# add_acq_id <- function(df){
#   df %>%
#     left_join(acquisitions %>% select(acq_id, all_of(common_cols)),
#               by = common_cols) %>%
#     relocate(acq_id, .before = everything())
# }

# # SNR results
# results_snr <- df_snr %>%
#   select(-any_of(common_cols)) %>%
#   bind_cols(df_snr %>% select(all_of(common_cols))) %>%
#   add_acq_id() %>%
#   select(acq_id, #any_of(angular_atm_cols), 
#          Signal_to_noise_ratio,
#          TOA_reflectance_at_normalized_view_geometry)

# # AbsCal results
# results_abscal <- df_abs %>%
#   select(-any_of(common_cols)) %>%
#   bind_cols(df_abs %>% select(all_of(common_cols))) %>%
#   add_acq_id() %>%
#   select(acq_id, #any_of(angular_atm_cols),
#          TOA_reflectance_at_normalized_view_geometry,
#          TOA_reflectance_at_normalized_view_geometry_reference,
#          Gain_TOA_reflectance,
#          Surface_reflectance,
#          Surface_reflectance_reference1,
#          Surface_reflectance_reference2,
#          Gain_surface_reflectance_1,
#          Gain_surface_reflectance_2)

# # TempStab results
# results_tempstab <- df_tempstab %>%
#   select(-any_of(common_cols)) %>%
#   bind_cols(df_tempstab %>% select(all_of(common_cols))) %>%
#   add_acq_id() %>%
#   select(acq_id,
#          Time_series_date_min,
#          Time_series_date_max,
#          Number_of_images,
#          `Time_series_length(days)`,
#          starts_with('Radiometric_Temporal_Stability'))

# # # ============================================================
# # # SAVE TO SQLITE DATABASE
# # # ============================================================
# # db_path <- 'radiometric_eval_db.sqlite'
# # con <- dbConnect(SQLite(), db_path)

# # # Write tables
# # dbWriteTable(con, 'acquisitions',    acquisitions,    overwrite = TRUE)
# # dbWriteTable(con, 'results_snr',     results_snr,     overwrite = TRUE)
# # dbWriteTable(con, 'results_abscal',  results_abscal,  overwrite = TRUE)
# # dbWriteTable(con, 'results_tempstab',results_tempstab, overwrite = TRUE)

# # # Add indexes for fast joins
# # dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_acq_id_snr      ON results_snr(acq_id)')
# # dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_acq_id_abscal   ON results_abscal(acq_id)')
# # dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_acq_id_tempstab ON results_tempstab(acq_id)')
# # dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_satellite        ON acquisitions(Satellite)')
# # dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_eval_type        ON acquisitions(Evaluation_type)')

# # dbDisconnect(con)

# # print(glue::glue("Database saved to: {db_path}"))
# # print(glue::glue("Tables: acquisitions ({nrow(acquisitions)}), 
# #                   SNR ({nrow(results_snr)}), 
# #                   AbsCal ({nrow(results_abscal)}), 
# #                   TempStab ({nrow(results_tempstab)})"))